In [25]:
import pandas as pd
import numpy as np
import openpyxl
import locale
import sys
sys.path.append('../')
from src import sp_eda as se
pd.set_option('display.max_columns', None)

Insertamos el dataframe y lo copiamos

In [26]:
df_raw=pd.read_csv('../data/Acc_met.csv')

df=df_raw.copy()
df.sample(5)

,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK,Name
6077,LA,2022-11-07,69.0,0.0,69.0,0.0,0.0,162.0,10.0,122.0,0.0,0.0,30.0,0.0,0.0,0.0,30.0,4.0,0.0,128.0,0.0,Louisiana
14337,SD,2022-10-16,NaN,NaN,NaN,NaN,NaN,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,South Dakota
4589,IL,2022-10-03,29.0,0.0,28.0,0.0,1.0,29.0,0.0,28.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,5.0,0.0,23.0,0.0,Illinois
4639,IL,2022-11-22,66.0,0.0,65.0,0.0,1.0,8.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,Illinois
205,AL,2022-07-25,91.0,4.0,63.0,23.0,1.0,50.0,3.0,8.0,0.0,8.0,31.0,0.0,0.0,5.0,22.0,4.0,0.0,11.0,8.0,Alabama


Movemos la columna de posición y cambiamos de nombre la columna con el codigo de estado.

In [27]:
col = "Name"
cols = list(df.columns)
cols.insert(1, cols.pop(cols.index(col)))
df= df[cols]
df = df.rename(columns={
    "State": "State_code",
    "Name": "State"
})



df.sample(5)

,State_code,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK
10607,NJ,New Jersey,2022-05-22,53.0,0.0,52.0,1.0,0.0,12.0,0.0,7.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,7.0,0.0
4032,ID,Idaho,2022-03-20,NaN,NaN,NaN,NaN,NaN,95.0,0.0,9.0,0.0,0.0,29.0,53.0,4.0,4.0,65.0,14.0,0.0,12.0,0.0
14866,TX,Texas,2022-04-09,174.0,0.0,145.0,29.0,0.0,36.0,5.0,28.0,0.0,0.0,3.0,0.0,0.0,0.0,3.0,24.0,0.0,9.0,0.0
17440,WY,Wyoming,2022-06-02,3.0,0.0,3.0,0.0,0.0,17.0,0.0,6.0,0.0,0.0,11.0,0.0,0.0,0.0,11.0,0.0,0.0,6.0,0.0
1842,CT,Connecticut,2022-01-26,42.0,0.0,32.0,8.0,2.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0


In [28]:
se.eda_prelim(df)

----------
DIMENSIONES
El conjunto de datos presenta 17653 filas y 22 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 17653 entries, 0 to 17652
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   State_code         17653 non-null  str    
 1   State              17653 non-null  str    
 2   Date               17653 non-null  str    
 3   acc_events         14431 non-null  float64
 4   acc_sev_1          14431 non-null  float64
 5   acc_sev_2          14431 non-null  float64
 6   acc_sev_3          14431 non-null  float64
 7   acc_sev_4          14431 non-null  float64
 8   weather_events     16714 non-null  float64
 9   typ_Cold           16714 non-null  float64
 10  typ_Fog            16714 non-null  float64
 11  typ_Hail           16714 non-null  float64
 12  typ_Precipitation  16714 non-null  float64
 13  typ_Rain           16714 non-null  float64
 14  typ_Snow       

Se van a unificar el tipo de datos de las columnas y vamos a usar tipo int, ya que para contar eventos nos es más que suficiente y es más eficiente.
En las filas de severidad de eventos meteorologicos, vamos a sustituir NaN por 0.

In [29]:

cols = df.columns[3:21]

df[cols] = df[cols].fillna(0).astype("int64")


Por otro lado, en la clasificación de eventos meteorologicos, existen dos tipos que vamos a unificar, ya que la información que aportan por separado no aporta más que toda ella junta (other y unknown).


In [30]:
df["wea_sev_UNKNOWN"] = (
    df[["wea_sev_UNK", "wea_sev_Other"]]
    .fillna(0)
    .sum(axis=1)
    .astype("Int64")
)

df = df.drop(columns=["wea_sev_UNK", "wea_sev_Other"])

Ponemos los nombres de las columnas todas en mayúsculas por estandarizar

In [31]:
df.columns = df.columns.str.upper()


Ordeno las columnas por severidad de evento meteorologico

In [32]:

old_pos = 16   # columna 16
new_pos = 18   # columna 18

cols = list(df.columns)
col = cols.pop(old_pos)
cols.insert(new_pos, col)
df = df[cols]


Vamos a cambiar el nombre de la severidad de accidentes, de 1, 2 3 y 4 a LIGHT, MODERATE, HEAVY y SEVERE respectivamente

In [33]:

cols_to_change = df.columns[4:8]

replacements = {
    "1": "LIGHT",
    "2": "MODERATE",
    "3": "HEAVY",
    "4": "SEVERE"
}
new_names = [
    col.replace("1", "LIGHT")
       .replace("2", "MODERATE")
       .replace("3", "HEAVY")
       .replace("4", "SEVERE")
    for col in cols_to_change
]
df.rename(columns=dict(zip(cols_to_change, new_names)), inplace=True)


Con esto damos por concluida la limpieza de datos y procedemos a guardar nuestro df 

In [34]:
df.to_csv('../data/Acc_met_clean.csv', index=False)

In [35]:
se.eda_prelim(df)
df.sample(5)

----------
DIMENSIONES
El conjunto de datos presenta 17653 filas y 21 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 17653 entries, 0 to 17652
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   STATE_CODE         17653 non-null  str  
 1   STATE              17653 non-null  str  
 2   DATE               17653 non-null  str  
 3   ACC_EVENTS         17653 non-null  int64
 4   ACC_SEV_LIGHT      17653 non-null  int64
 5   ACC_SEV_MODERATE   17653 non-null  int64
 6   ACC_SEV_HEAVY      17653 non-null  int64
 7   ACC_SEV_SEVERE     17653 non-null  int64
 8   WEATHER_EVENTS     17653 non-null  int64
 9   TYP_COLD           17653 non-null  int64
 10  TYP_FOG            17653 non-null  int64
 11  TYP_HAIL           17653 non-null  int64
 12  TYP_PRECIPITATION  17653 non-null  int64
 13  TYP_RAIN           17653 non-null  int64
 14  TYP_SNOW           17653 non-null  int64
 15  T

,STATE_CODE,STATE,DATE,ACC_EVENTS,ACC_SEV_LIGHT,ACC_SEV_MODERATE,ACC_SEV_HEAVY,ACC_SEV_SEVERE,WEATHER_EVENTS,TYP_COLD,TYP_FOG,TYP_HAIL,TYP_PRECIPITATION,TYP_RAIN,TYP_SNOW,TYP_STORM,WEA_SEV_LIGHT,WEA_SEV_MODERATE,WEA_SEV_HEAVY,WEA_SEV_SEVERE,WEA_SEV_UNKNOWN
927,AZ,Arizona,2022-07-23,36,0,35,1,0,48,0,11,0,1,35,0,1,32,5,3,7,1
329,AL,Alabama,2022-11-27,0,0,0,0,0,231,0,7,0,6,218,0,0,152,51,15,7,6
15527,VA,Virginia,2022-02-05,140,0,120,7,13,101,0,7,0,0,53,41,0,82,18,0,1,0
3141,FL,Florida,2022-10-08,262,0,259,0,3,34,3,16,0,0,15,0,0,14,4,0,16,0
16010,VT,Vermont,2022-06-11,0,0,0,0,0,5,0,3,0,0,2,0,0,2,1,0,2,0
